# vulcan-jwst-tool: programmatic API walkthrough

The Streamlit GUI (`jwst-tool` console script) is the primary interface; this notebook shows the same machinery driven directly from Python: the planet/instrument registries, the forward-model cache, detection significance, and the Fisher forecast.

**Prerequisites**: `pip install --no-deps -e ./vulcan-jwst-tool` (plus vulcan-retrieval and vulcan-jax) in a checkout of the `vulcan_exojax_run` repo. The registry/detect/fisher cells are cheap pure-numpy; only the (optional) live forward-model and Pandeia cells are heavy, and the Pandeia backend additionally needs its own conda env (`JWST_TOOL_PANDEIA_PYTHON`, see the README).

In [ ]:
import numpy as np
from jwst_tool import planets, instruments, detect, fisher, forward

print('planets:', ', '.join(planets.PLANETS))
print('modes  :', ', '.join(instruments.MODES))
print('data   :', instruments.DATA_DIR)

## 1. Registries

`planets.PLANETS` carries the star/planet/system identity (plus the stellar-UV choice for the photochemistry); `instruments.MODES` carries the seven JWST mode definitions with their Pandeia configs, wavelength coverage, and noise floors.

In [ ]:
p = planets.PLANETS['wasp39b']
print(p['label'], '| Teq', p['teq_k'], 'K | t14', p['t14_hr'], 'hr | UV:', p['sflux'])
for key, m in instruments.MODES.items():
    print(f"{key:18s} {m['label']:16s} {m['wl_min']:.2f}-{m['wl_max']:.2f} um  floor {m['floor_ppm']} ppm")

## 2. Forward-model spectra (cache face)

`forward.load_result(params)` reads the npz cache the GUI populates; it returns `None` for an uncached parameter set. Generating a NEW spectrum runs the live VULCAN-JAX + ExoJax chain (about 2 to 3 minutes per parameter set) either from the GUI or headlessly:

```bash
python vulcan-jwst-tool/src/jwst_tool/forward.py params.json   # heavy script face
```

The cache key includes `forward._VERSION`, so stale physics never leaks through a version bump.

In [ ]:
params = forward.canonical_params({'planet': 'wasp39b'})
res = forward.load_result(params)
if res is None:
    print('no cached spectrum for the default WASP-39b parameter set yet;')
    print('generate one via the GUI (jwst-tool) or the script face above.')
else:
    import matplotlib.pyplot as plt
    print('cached arrays:', sorted(res))
    plt.figure(figsize=(7, 3))
    plt.plot(res['wl_um'], res['depth'] * 1e6, lw=0.7)
    plt.xlabel('wavelength (um)'); plt.ylabel('transit depth (ppm)')
    plt.title(', '.join(np.atleast_1d(res['mols']).tolist()))
    plt.tight_layout()

## 3. Detection significance (pure numpy)

`detect.detection_significance` is the offset-marginalized `sqrt(Delta chi^2)` of a binned molecular signal against per-bin noise: the statistic the tool ranks instrument modes by. A toy example with a synthetic 50 ppm feature over 20 bins:

In [ ]:
rng = np.random.default_rng(0)
signal = 50e-6 * np.exp(-0.5 * ((np.arange(20) - 10) / 3.0) ** 2)   # ppm-scale feature
sigma = np.full(20, 30e-6)
print('raw quadrature :', detect.detection_significance(signal, sigma, marginalize_offset=False))
print('offset-marg.   :', detect.detection_significance(signal, sigma))   # the honest number
print('(offset marginalization removes the part of the signal a free baseline absorbs)')

## 4. Fisher forecast

When a cached spectrum was generated with Fisher requested, it carries the autodiff Jacobian (`jac`, `jac_names`). `fisher.mode_forecast` turns one mode's binned Jacobian + noise into marginalized parameter sigmas (with the shared lnR0 and per-mode offset nuisances), and `fisher.combined_forecast` stacks several modes. These consume the `result` dicts the GUI assembles per mode; see `fisher.py` docstrings for the exact dict shape if you want to build them by hand.

In [ ]:
if res is not None and 'jac' in res:
    print('Jacobian rows:', list(res['jac_names']))
    print('shape:', res['jac'].shape)
else:
    print('cached spectrum has no Jacobian (generate with Fisher enabled in the GUI)')

## 5. Noise (Pandeia) backend

Real per-bin noise comes from the STScI Pandeia ETC, which runs in its own conda env: `noise.py` shells out to `pandeia_worker.py` via `JWST_TOOL_PANDEIA_PYTHON`, and results land in the `noise_cache/`. That backend is deliberately not exercised here; the GUI (or `noise.py`'s functions with the env configured) drives it. See the README's backend-wiring section.